# model-diff v0 — Batch comparison (7 model pairs)

**Goal:** Run weight delta analysis on multiple base→instruct pairs to find generalizable patterns.

- **Runtime:** CPU (12.7 GB, 0.07 units/hr) — memory-optimized SVD fits in standard RAM
- **Disk:** ~30 GB per 7B pair (base + instruct); HF cache cleaned between pairs to stay under 250 GB
- **Expected runtime:** ~20 min per pair × 7 pairs = ~2.5 hours
- **Output:** JSON per pair + cross-model comparison plots

In [ ]:
!pip install -q safetensors huggingface_hub torch numpy matplotlib

from google.colab import drive, userdata
drive.mount('/content/drive')

import os
HF_TOKEN = userdata.get('HF_TOKEN')
os.environ['HF_TOKEN'] = HF_TOKEN or ''

DRIVE_DIR = '/content/drive/MyDrive/model_diff'
for sub in ['results', 'reports']:
    os.makedirs(f'{DRIVE_DIR}/{sub}', exist_ok=True)

print('Drive root:', DRIVE_DIR)

In [ ]:
# ── Model pairs to compare ─────────────────────────────────────

PAIRS = [
    # (base, instruct, short_name)
    ('Qwen/Qwen2.5-7B', 'Qwen/Qwen2.5-7B-Instruct', 'qwen2.5_7b'),
    ('meta-llama/Llama-3.1-8B', 'meta-llama/Llama-3.1-8B-Instruct', 'llama3.1_8b'),
    ('mistralai/Mistral-7B-v0.3', 'mistralai/Mistral-7B-Instruct-v0.3', 'mistral_7b_v03'),
    ('google/gemma-2-9b', 'google/gemma-2-9b-it', 'gemma2_9b'),
    ('microsoft/Phi-3.5-mini-instruct', 'microsoft/Phi-3.5-mini-instruct', 'phi3.5_mini'),  # placeholder — update if base exists
    ('Qwen/Qwen2.5-3B', 'Qwen/Qwen2.5-3B-Instruct', 'qwen2.5_3b'),
    ('Qwen/Qwen2.5-14B', 'Qwen/Qwen2.5-14B-Instruct', 'qwen2.5_14b'),
    ('meta-llama/Llama-3.2-3B', 'meta-llama/Llama-3.2-3B-Instruct', 'llama3.2_3b'),
]

# Remove placeholder pairs (same model = no delta)
PAIRS = [(a, b, n) for a, b, n in PAIRS if a != b]
print(f'{len(PAIRS)} pairs to analyze')
for a, b, n in PAIRS:
    print(f'  {n}: {a} → {b}')

In [ ]:
import gc
import json
import math
import shutil
import time
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
from huggingface_hub import snapshot_download
from safetensors import safe_open


# ── Core functions (memory-optimized) ──────────────────────────

def download_safetensors(model_id):
    path = snapshot_download(
        model_id,
        allow_patterns=['*.safetensors', '*.safetensors.index.json', 'config.json'],
        token=os.environ.get('HF_TOKEN'),
    )
    return path


def get_tensor_map(model_path):
    index_file = os.path.join(model_path, 'model.safetensors.index.json')
    if os.path.exists(index_file):
        with open(index_file) as f:
            index = json.load(f)
        return {k: os.path.join(model_path, v) for k, v in index['weight_map'].items()}
    sf_file = os.path.join(model_path, 'model.safetensors')
    if not os.path.exists(sf_file):
        raise FileNotFoundError(f'No safetensors found in {model_path}')
    with safe_open(sf_file, framework='pt') as f:
        return {name: sf_file for name in f.keys()}


def load_tensor(tensor_map, name):
    shard_path = tensor_map[name]
    with safe_open(shard_path, framework='pt', device='cpu') as f:
        return f.get_tensor(name)


def effective_rank(sigma):
    s = sigma / sigma.sum()
    s = s[s > 1e-12]
    return float(torch.exp(-(s * torch.log(s)).sum()))


def compute_cosine_sim(a, b):
    cos = torch.nn.functional.cosine_similarity(a.unsqueeze(0), b.unsqueeze(0)).item()
    return float(max(-1.0, min(1.0, cos)))


def analyze_delta(w_a, w_b, top_k=20, sparsity_threshold=1e-5):
    """Memory-optimized: frees w_a/w_b before SVD. Peak ≈ 2x tensor size."""
    shape = w_a.shape
    n_params = int(w_a.numel())

    if w_a.ndim == 1:
        delta = (w_b - w_a).float()
        frob = float(delta.norm())
        return {
            'shape': list(shape), 'n_params': n_params,
            'frob_norm': frob,
            'frob_norm_relative': frob / (float(w_a.float().norm()) + 1e-12),
            'cosine_sim': compute_cosine_sim(w_a.float().flatten(), w_b.float().flatten()),
            'sparsity': float((delta.abs() < sparsity_threshold).float().mean()),
            'has_svd': False,
        }

    # ── Phase 1: metrics that need both w_a and w_b ───────────
    w_a_2d = w_a.float().reshape(shape[0], -1)
    w_b_2d = w_b.float().reshape(shape[0], -1)

    norm_a = float(w_a_2d.norm())
    cos_sim = compute_cosine_sim(w_a_2d.flatten(), w_b_2d.flatten())

    # In-place subtraction: w_b_2d becomes delta, no third large tensor
    delta_2d = w_b_2d.sub_(w_a_2d)

    frob = float(delta_2d.norm())
    frob_rel = frob / (norm_a + 1e-12)
    sparsity = float((delta_2d.abs() < sparsity_threshold).float().mean())

    # ── Free w_a, w_b before SVD (biggest RAM saver) ──────────
    del w_a_2d, w_a, w_b

    # ── Phase 2: SVD on delta only ────────────────────────────
    m, n = delta_2d.shape
    if min(m, n) > 8192:
        k = min(top_k * 2, min(m, n))
        Q, _ = torch.linalg.qr(delta_2d @ torch.randn(n, k))
        B = Q.T @ delta_2d
        del Q
        _, sigma, _ = torch.linalg.svd(B, full_matrices=False)
        del B
        sigma = sigma[:top_k]
    else:
        sigma = torch.linalg.svdvals(delta_2d)

    del delta_2d

    eff_rank = effective_rank(sigma)
    top_k_actual = min(top_k, len(sigma))
    top_sigmas = sigma[:top_k_actual]
    total_energy = float((sigma**2).sum())
    top_k_energy = float((top_sigmas**2).sum())
    concentration = top_k_energy / (total_energy + 1e-12)

    log_sigma = torch.log(sigma[sigma > 1e-12])
    log_ranks = torch.log(torch.arange(1, len(log_sigma) + 1).float())
    if len(log_sigma) > 2:
        A = torch.stack([log_ranks, torch.ones_like(log_ranks)], dim=1)
        result = torch.linalg.lstsq(A, log_sigma)
        spectral_alpha = float(-result.solution[0])
    else:
        spectral_alpha = None

    return {
        'shape': list(shape), 'n_params': n_params,
        'frob_norm': frob, 'frob_norm_relative': frob_rel,
        'cosine_sim': cos_sim, 'sparsity': sparsity,
        'has_svd': True, 'effective_rank': eff_rank,
        'concentration_top_k': concentration, 'top_k': top_k_actual,
        'top_singular_values': top_sigmas.tolist(),
        'spectral_alpha': spectral_alpha, 'n_singular_values': int(len(sigma)),
    }


def cleanup_hf_cache(model_ids):
    """Remove downloaded model files from HF cache to free disk space."""
    cache_dir = Path.home() / '.cache' / 'huggingface' / 'hub'
    if not cache_dir.exists():
        return
    freed = 0
    for model_id in model_ids:
        # HF cache uses models--org--name format
        cache_name = 'models--' + model_id.replace('/', '--')
        model_cache = cache_dir / cache_name
        if model_cache.exists():
            size = sum(f.stat().st_size for f in model_cache.rglob('*') if f.is_file())
            shutil.rmtree(model_cache)
            freed += size
    if freed > 0:
        print(f'  Freed {freed / 1e9:.1f} GB from HF cache')


print('Core functions loaded (memory-optimized)')

In [ ]:
# ── Run batch analysis ────────────────────────────────────────

all_results = {}  # short_name -> {model_a, model_b, modules, ...}

for pair_idx, (model_a, model_b, short_name) in enumerate(PAIRS):
    result_path = f'{DRIVE_DIR}/results/v0_{short_name}.json'

    # Skip if already computed
    if os.path.exists(result_path):
        print(f'\n[{pair_idx+1}/{len(PAIRS)}] {short_name}: CACHED, loading from {result_path}')
        with open(result_path) as f:
            all_results[short_name] = json.load(f)
        continue

    print(f'\n{"="*60}')
    print(f'[{pair_idx+1}/{len(PAIRS)}] {short_name}: {model_a} → {model_b}')
    print(f'{"="*60}')

    try:
        t0 = time.time()
        path_a = download_safetensors(model_a)
        path_b = download_safetensors(model_b)
        print(f'  Downloads: {time.time()-t0:.0f}s')

        tensor_map_a = get_tensor_map(path_a)
        tensor_map_b = get_tensor_map(path_b)

        names_a = set(tensor_map_a.keys())
        names_b = set(tensor_map_b.keys())
        common = sorted(names_a & names_b)
        print(f'  Common tensors: {len(common)}')

        results = []
        skipped = 0
        t1 = time.time()

        for i, name in enumerate(common):
            w_a = load_tensor(tensor_map_a, name)
            w_b = load_tensor(tensor_map_b, name)

            if w_a.numel() < 1000 or w_a.shape != w_b.shape:
                skipped += 1
                del w_a, w_b
                continue

            metrics = analyze_delta(w_a, w_b, top_k=20, sparsity_threshold=1e-5)
            del w_a, w_b  # caller refs must go too
            metrics['name'] = name
            results.append(metrics)

            gc.collect()

            if (i + 1) % 50 == 0 or (i + 1) == len(common):
                print(f'  [{i+1}/{len(common)}] {time.time()-t1:.0f}s')

        data = {
            'model_a': model_a, 'model_b': model_b,
            'short_name': short_name,
            'n_tensors': len(results), 'n_skipped': skipped,
            'modules': results,
        }

        with open(result_path, 'w') as f:
            json.dump(data, f, indent=2)
        print(f'  Saved: {result_path} ({len(results)} tensors, {time.time()-t0:.0f}s total)')

        all_results[short_name] = data

    except Exception as e:
        print(f'  ERROR: {e}')
        continue

    # Free disk: delete downloaded model files from HF cache
    del tensor_map_a, tensor_map_b
    cleanup_hf_cache([model_a, model_b])
    gc.collect()

print(f'\n\nBatch complete: {len(all_results)}/{len(PAIRS)} pairs analyzed')

In [ ]:
# ── Load all results (if restarting) ──────────────────────────

import glob

if not all_results:
    all_results = {}
    for f in sorted(glob.glob(f'{DRIVE_DIR}/results/v0_*.json')):
        with open(f) as fh:
            d = json.load(fh)
        name = d.get('short_name', Path(f).stem.replace('v0_', ''))
        all_results[name] = d
        print(f'Loaded {name}: {d["model_a"]} → {d["model_b"]} ({d["n_tensors"]} tensors)')

print(f'\nTotal: {len(all_results)} pairs loaded')

In [ ]:
# ── Cross-model comparison ────────────────────────────────────

import re

def module_type(name):
    """Extract module type (q_proj, mlp.gate, etc)."""
    for pat in ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj',
                'down_proj', 'lm_head', 'embed_tokens']:
        if pat in name:
            return pat
    if 'layernorm' in name.lower() or 'norm' in name.lower():
        return 'norm'
    return 'other'


def layer_idx(name):
    m = re.search(r'layers\.(\d+)\.', name)
    return int(m.group(1)) if m else None


# ── 1. Mean dW/W by module type across models ─────────────────
mod_types = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj',
             'down_proj', 'lm_head', 'embed_tokens', 'norm']

model_names = sorted(all_results.keys())
type_means = {}

for name in model_names:
    modules = all_results[name]['modules']
    type_means[name] = {}
    for mt in mod_types:
        vals = [m['frob_norm_relative'] for m in modules if module_type(m['name']) == mt]
        type_means[name][mt] = sum(vals) / len(vals) if vals else 0

# Print table
print(f'{"Module type":<15}', '  '.join(f'{n:>14}' for n in model_names))
print('─' * (15 + 16 * len(model_names)))
for mt in mod_types:
    vals = '  '.join(f'{type_means[n].get(mt, 0):>14.5f}' for n in model_names)
    print(f'{mt:<15} {vals}')


# ── 2. Bar chart: mean dW/W by module type ───────────────────
fig, ax = plt.subplots(figsize=(14, 6))
x = np.arange(len(mod_types))
width = 0.8 / len(model_names)

for i, name in enumerate(model_names):
    vals = [type_means[name].get(mt, 0) for mt in mod_types]
    ax.bar(x + i * width, vals, width, label=name)

ax.set_xticks(x + width * len(model_names) / 2)
ax.set_xticklabels(mod_types, rotation=45, ha='right')
ax.set_ylabel('Mean ||ΔW||/||W||')
ax.set_title('Mean relative delta by module type across models')
ax.legend(fontsize=8)
plt.tight_layout()
fig.savefig(f'{DRIVE_DIR}/reports/v0_batch_module_type_comparison.png', dpi=150)
plt.show()

In [ ]:
# ── 3. Effective rank distribution by module type ─────────────

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3a. Effective rank by module type
ax = axes[0, 0]
attn_types = ['q_proj', 'k_proj', 'v_proj', 'o_proj']
mlp_types = ['gate_proj', 'up_proj', 'down_proj']
for name in model_names:
    modules = all_results[name]['modules']
    attn_ranks = [m['effective_rank'] for m in modules if m.get('has_svd') and module_type(m['name']) in attn_types]
    mlp_ranks = [m['effective_rank'] for m in modules if m.get('has_svd') and module_type(m['name']) in mlp_types]
    ax.scatter([name]*len(attn_ranks), attn_ranks, alpha=0.3, s=10, c='blue', label='attn' if name == model_names[0] else '')
    ax.scatter([name]*len(mlp_ranks), mlp_ranks, alpha=0.3, s=10, c='red', label='mlp' if name == model_names[0] else '')
ax.set_ylabel('Effective rank')
ax.set_title('Effective rank: attention vs MLP')
ax.legend()
ax.tick_params(axis='x', rotation=45)

# 3b. Top-20 concentration by module type
ax = axes[0, 1]
for name in model_names:
    modules = all_results[name]['modules']
    attn_conc = [m['concentration_top_k'] for m in modules if m.get('has_svd') and module_type(m['name']) in attn_types]
    mlp_conc = [m['concentration_top_k'] for m in modules if m.get('has_svd') and module_type(m['name']) in mlp_types]
    ax.scatter([name]*len(attn_conc), attn_conc, alpha=0.3, s=10, c='blue')
    ax.scatter([name]*len(mlp_conc), mlp_conc, alpha=0.3, s=10, c='red')
ax.set_ylabel('Top-20 concentration')
ax.set_title('Concentration: attention vs MLP')
ax.tick_params(axis='x', rotation=45)

# 3c. Per-model summary: mean dW/W
ax = axes[1, 0]
mean_dw = [np.mean([m['frob_norm_relative'] for m in all_results[n]['modules']]) for n in model_names]
ax.bar(model_names, mean_dw, color='steelblue')
ax.set_ylabel('Mean ||ΔW||/||W||')
ax.set_title('Overall instruct tuning magnitude')
ax.tick_params(axis='x', rotation=45)

# 3d. Layer profile: normalized dW/W curve
ax = axes[1, 1]
for name in model_names:
    modules = all_results[name]['modules']
    layer_vals = {}
    for m in modules:
        li = layer_idx(m['name'])
        if li is not None:
            layer_vals.setdefault(li, []).append(m['frob_norm_relative'])
    if not layer_vals:
        continue
    max_layer = max(layer_vals.keys())
    # Normalize layer index to [0, 1] for cross-model comparison
    xs = [li / max_layer for li in sorted(layer_vals.keys())]
    ys = [np.mean(layer_vals[li]) for li in sorted(layer_vals.keys())]
    # Normalize magnitude to [0, 1] for shape comparison
    y_max = max(ys) if max(ys) > 0 else 1
    ys_norm = [y / y_max for y in ys]
    ax.plot(xs, ys_norm, label=name, alpha=0.7)
ax.set_xlabel('Relative layer position')
ax.set_ylabel('Normalized mean ||ΔW||/||W||')
ax.set_title('Layer-wise delta profile (normalized)')
ax.legend(fontsize=7)

plt.tight_layout()
fig.savefig(f'{DRIVE_DIR}/reports/v0_batch_cross_model.png', dpi=150)
plt.show()
print(f'Saved to {DRIVE_DIR}/reports/v0_batch_cross_model.png')

In [ ]:
# ── 4. Summary table for paper ───────────────────────────────

print(f'{"Model pair":<25} {"N_tens":>7} {"Mean dW/W":>10} {"Max dW/W":>10} {"Mean cos":>10} {"Mean rank":>10} {"Mean conc":>10}')
print('─' * 85)

for name in model_names:
    modules = all_results[name]['modules']
    svd_mods = [m for m in modules if m.get('has_svd')]
    dw = [m['frob_norm_relative'] for m in modules]
    cos_vals = [min(1.0, m['cosine_sim']) for m in modules]
    ranks = [m['effective_rank'] for m in svd_mods if m.get('effective_rank')]
    concs = [m['concentration_top_k'] for m in svd_mods if m.get('concentration_top_k')]

    print(f'{name:<25} {len(modules):>7} {np.mean(dw):>10.5f} {max(dw):>10.5f} '
          f'{np.mean(cos_vals):>10.5f} {np.mean(ranks):>10.0f} {np.mean(concs):>10.3f}')

print('\n\n=== Module type ranking (most changed → least changed) ===')
# Average across all models
avg_by_type = {}
for mt in mod_types:
    vals = [type_means[n].get(mt, 0) for n in model_names]
    avg_by_type[mt] = np.mean(vals)

for mt, v in sorted(avg_by_type.items(), key=lambda x: x[1], reverse=True):
    print(f'  {mt:<15} {v:.5f}')